In [2]:
pip install sdv sdmetrics

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   ------------------

# LIBRERIA SDV
# Autor: Fernanda Pacheco Banda
# Fecha: 04 de junio

In [1]:
#Importamos Librerias
import pandas as pd
import numpy as np
import random

In [2]:
import sdv #Generar datos sinteticos
import sdmetrics #Evaluar la calidad de los datos
from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

In [3]:
from sdmetrics.reports.single_table import QualityReport

In [4]:
print ("SDV:", sdv.__version__)

SDV: 1.37.0


In [5]:
#Creamos un dataset original que servira como fuente para crear los datos sinteticos
dfClientes = pd.DataFrame(
    {
        "cliente_id": [1,2,3,4,5,6,7,8,9,10],
        "edad": [23,33,43,28,53,56,65,40,25,46],
        "ingreso_mensual": [25000,15000,20000,10000,5000,17000,30000,12000,35000,7500],
        "ciudad": ["Veracruz","Cordoba","Paso del Macho","Amatlan","Fortin","Cuitlahuac","Yanga","Cordoba","Orizaba","Cuitlahuac"]
    }
)

In [6]:
dfClientes.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,1,23,25000,Veracruz
1,2,33,15000,Cordoba
2,3,43,20000,Paso del Macho
3,4,28,10000,Amatlan
4,5,53,5000,Fortin


In [7]:
#Definir los metadatos
metadata = SingleTableMetadata()

In [8]:
metadata.detect_from_dataframe(
    data = dfClientes
)

In [ ]:
metadata.visualize()

In [10]:
metadata.to_dict()

{'primary_key': 'cliente_id',
 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1',
 'columns': {'cliente_id': {'sdtype': 'id'},
  'edad': {'sdtype': 'numerical'},
  'ingreso_mensual': {'sdtype': 'numerical'},
  'ciudad': {'sdtype': 'categorical'}}}

In [ ]:
#
metadata.save_to_json(
    "dfClientes_metadata.json"
)

In [12]:
#Entrenamos el modelo para generar los datos sinteticos
synthesizer = GaussianCopulaSynthesizer(
    metadata
)

C:\Users\pache\anaconda3\envs\Extraccion-9BESC\Lib\site-packages\sdv\single_table\base.py:182: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\pache\anaconda3\envs\Extraccion-9BESC\Lib\site-packages\sdv\single_table\base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Task was destroyed but it is pending!
task: <Task pending name='Task-102' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\pache\anaconda3\envs\Extraccion-9BESC\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-103' coro=<Kernel.shell_main() running at C:\Users\pache\anaconda3\envs\Extraccion-9BESC\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\pache\anaconda3\envs\Extraccion

In [13]:
#Entrenamiento 
synthesizer.fit(
    dfClientes
)

In [14]:
#Generamos los datos sinteticos
clientes_sinteticos = synthesizer.sample(
    num_rows=100
)

In [15]:
clientes_sinteticos.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,16169768,48,13388,Cordoba
1,4918803,48,20305,Yanga
2,1900081,35,20376,Amatlan
3,531516,33,17474,Yanga
4,10211768,51,10877,Amatlan


In [16]:
clientes_sinteticos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   cliente_id       100 non-null    int64 
 1   edad             100 non-null    int64 
 2   ingreso_mensual  100 non-null    int64 
 3   ciudad           100 non-null    object
dtypes: int64(3), object(1)
memory usage: 3.3+ KB


In [18]:
clientes_sinteticos.describe(include="all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.000000e+02,100.000000,100.000000,100
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,30
mean,8.532170e+06,42.570000,22582.030000,NaN
std,4.687773e+06,11.737237,8954.955259,NaN
min,4.660500e+04,25.000000,5951.000000,NaN
25%,5.008268e+06,32.000000,15070.750000,NaN
50%,8.656048e+06,42.000000,21953.000000,NaN
75%,1.256688e+07,51.500000,31727.250000,NaN


In [19]:
#DataFrame contaminado
dfClientesGIGO = clientes_sinteticos.copy()

In [20]:
#Colocar edades imposibles
indices = random.sample(list(dfClientesGIGO.index),5)

In [22]:
dfClientesGIGO.loc[indices,"edad"]= -5

In [23]:
#Introducimos valores duplicados
duplicados = dfClientesGIGO.sample(10,random_state=42)

In [24]:
dfClientesGIGO = pd.concat([dfClientesGIGO,duplicados], ignore_index=True)

In [30]:
#Verificamos valores nulos
dfClientesGIGO.duplicated().sum()

np.int64(10)

In [34]:
dfClientesGIGO.describe(include="all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.100000e+02,110.000000,110.000000,110
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,32
mean,8.614045e+06,39.554545,23205.372727,NaN
std,4.775996e+06,15.302807,8989.560842,NaN
min,4.660500e+04,-5.000000,5951.000000,NaN
25%,4.948624e+06,29.000000,16088.000000,NaN
50%,8.956428e+06,39.000000,22874.000000,NaN
75%,1.274153e+07,51.000000,32271.500000,NaN


In [37]:
#Generamos el reporte
reporte = QualityReport()

In [39]:
reporte.generate(
    real_data = dfClientes,
    synthetic_data = clientes_sinteticos,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 244.92it/s]|
Column Shapes Score: 78.0%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 87.93it/s]|
Column Pair Trends Score: 10.0%

Overall Score (Average): 44.0%



In [40]:
reporte.generate(
    real_data = dfClientes,
    synthetic_data = dfClientesGIGO,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 453.71it/s]|
Column Shapes Score: 77.88%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 147.89it/s]|
Column Pair Trends Score: 13.18%

Overall Score (Average): 45.53%

